# Pokemon TCG AI Battle Challenge - Lucario v2 Strategic Baseline

## Strategic Overview

Pokemon TCG is an imperfect-information stochastic game. The opponent hand, deck order, and prize cards are hidden, so turn-level play is best treated as legal-option ranking under uncertainty rather than fixed-script execution.

This agent uses Mega Lucario ex as a stable tactical core: a clear primary attacker, meaningful backup attackers, and enough trainer density to recover from weak openings. The policy emphasizes setup reliability, attack planning, and avoiding harmful optional choices.

In [1]:
from pathlib import Path
import glob
import json
import math
import os
import random
import sys
import shutil
import tarfile
import textwrap

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = print

SEED = 20260617
N_SMOKE_GAMES = 40
N_MATCHUP_GAMES = 100
LOW_DECK_COUNT = 8
USE_SEARCH = False
WRITE_SUBMISSION = True
REQUIRE_CG_IN_ARCHIVE = True
ENFORCE_GATE = True            # if True, a failed validation gate raises and blocks submission
WINRATE_GATE = 0.80            # min win-rate vs legal-random (alternating seat)
FALLBACK_GATE = 0.05           # max fraction of decisions allowed to hit the safety fallback
SEARCH_FAIL_GATE = 0.05        # max search failure rate when USE_SEARCH_IN_AGENT is enabled
MAX_STEPS_PER_GAME = 2000      # hard cap so a stuck game cannot hang validation
USE_SEARCH_IN_AGENT = False    # OFF by default; enable only after search A/B passes the failure and win-rate gates
SEARCH_PROFILE = "live"         # "live" keeps current 800+ settings; "safe" lowers search cost
SEARCH_NODE_BUDGET = 120       # max engine forward-steps per decision
SEARCH_TIME_BUDGET_S = 1.5     # wall-clock cap per decision (respect the 10-min/game limit)
SEARCH_ACTION_CAP = 20         # max root MAIN action combinations evaluated by search
if SEARCH_PROFILE == "safe":
    SEARCH_NODE_BUDGET = min(SEARCH_NODE_BUDGET, 80)
    SEARCH_TIME_BUDGET_S = min(SEARCH_TIME_BUDGET_S, 1.0)
    SEARCH_ACTION_CAP = min(SEARCH_ACTION_CAP, 12)
DECK_VARIANT = "tempo"         # options: baseline, tempo, keep_hero_tempo, energy_consistency

ROOT = Path.cwd()
SAMPLE_DECK_DIR = ROOT / "Sample_Decks"
PUBLIC_EXAMPLES_DIR = ROOT / "Public_Examples"
BUILD_DIR = ROOT / "phase1_lucario_v2_build"
SUBMISSION_PATH = ROOT / "submission.tar.gz"

random.seed(SEED)
if np is not None:
    np.random.seed(SEED)

print("ROOT:", ROOT)
print("Sample decks:", SAMPLE_DECK_DIR.exists(), SAMPLE_DECK_DIR)
print("Public examples:", PUBLIC_EXAMPLES_DIR.exists(), PUBLIC_EXAMPLES_DIR)
print("Require cg/ in final archive:", REQUIRE_CG_IN_ARCHIVE)




ROOT: /kaggle/working
Sample decks: False /kaggle/working/Sample_Decks
Public examples: False /kaggle/working/Public_Examples
Require cg/ in final archive: True


## Deck Engine

Core plan:

- **Riolu -> Mega Lucario ex** is the main pressure line.
- **Mega Brave** supplies the high-damage closing line.
- **Hariyama** and **Solrock** keep the deck functional when Lucario is delayed or overexposed.
- **Dusk Ball**, **Poke Pad**, **Carmine**, and **Lillie's Determination** support setup consistency.
- **Switch**, **Boss's Orders**, and **Gravity Mountain** help convert tactical attack plans into knockouts.

The deck is built around consistency first, then tempo, then selective finishing pressure. The Keep-Hero tempo branch preserves Boss/Riolu pressure while retaining the defensive Hero Cape line for matchup testing.

In [2]:
BASELINE_DECK = [
    673, 673, 674, 674, 675, 675, 676, 676,
    676, 677, 677, 677, 678, 678, 678, 678,
    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,
    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,
    1152, 1152, 1159, 1182, 1182, 1192, 1192, 1192,
    1192, 1227, 1227, 1227, 1227, 1252, 1252, 6,
    6, 6, 6, 6, 6, 6, 6, 6,
    6, 6, 6, 6,
]

CARD_LABELS = {
    673: ("Makuhita", "Pokemon", "Basic"),
    674: ("Hariyama", "Pokemon", "Stage 1"),
    675: ("Lunatone", "Pokemon", "Basic"),
    676: ("Solrock", "Pokemon", "Basic"),
    677: ("Riolu", "Pokemon", "Basic"),
    678: ("Mega Lucario ex", "Pokemon", "Mega ex"),
    1102: ("Dusk Ball", "Trainer", "Item"),
    1123: ("Switch", "Trainer", "Item"),
    1141: ("Premium Power Pro", "Trainer", "Tool"),
    1142: ("Fighting Gong", "Trainer", "Item"),
    1152: ("Poke Pad", "Trainer", "Item"),
    1159: ("Hero Cape", "Trainer", "Tool"),
    1182: ("Boss's Orders", "Trainer", "Supporter"),
    1192: ("Carmine", "Trainer", "Supporter"),
    1227: ("Lillie's Determination", "Trainer", "Supporter"),
    1252: ("Gravity Mountain", "Trainer", "Stadium"),
    6: ("Basic Fighting Energy", "Energy", "Basic Energy"),
}

from collections import Counter

assert len(BASELINE_DECK) == 60
assert all(isinstance(x, int) for x in BASELINE_DECK)
assert set(BASELINE_DECK) <= set(CARD_LABELS), "Every deck card should have a notebook label."

# Deck variants use only cards already understood by the policy.
# tempo: +Boss/+Riolu, -Gravity/-HeroCape for mirror pressure.
# keep_hero_tempo: keeps the defensive Hero Cape out while trimming Poke Pad instead.
# energy_consistency: adds a Basic Energy and Riolu, trading off a search card and spare Stadium.

def _apply(deck, card_id, delta):
    d = list(deck)
    if delta < 0:
        for _ in range(-delta):
            d.remove(card_id)
    else:
        d.extend([card_id] * delta)
    return d

TEMPO_DECK = list(BASELINE_DECK)
TEMPO_DECK = _apply(TEMPO_DECK, 1182, +1)
TEMPO_DECK = _apply(TEMPO_DECK, 677, +1)
TEMPO_DECK = _apply(TEMPO_DECK, 1252, -1)
TEMPO_DECK = _apply(TEMPO_DECK, 1159, -1)

KEEP_HERO_TEMPO_DECK = list(BASELINE_DECK)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1182, +1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 677, +1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1252, -1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1152, -1)

ENERGY_CONSISTENCY_DECK = list(BASELINE_DECK)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 677, +1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 6, +1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 1252, -1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 1152, -1)

DECK_VARIANTS = {
    "baseline": BASELINE_DECK,
    "tempo": TEMPO_DECK,
    "keep_hero_tempo": KEEP_HERO_TEMPO_DECK,
    "energy_consistency": ENERGY_CONSISTENCY_DECK,
}
assert DECK_VARIANT in DECK_VARIANTS, sorted(DECK_VARIANTS)
DECK = DECK_VARIANTS[DECK_VARIANT]
counts = Counter(DECK)
basic_ids = {673, 675, 676, 677}
basic_count = sum(counts[i] for i in basic_ids)
prob_no_basic_opening_7 = math.comb(len(DECK) - basic_count, 7) / math.comb(len(DECK), 7)

assert len(DECK) == 60, len(DECK)
assert set(DECK) <= set(CARD_LABELS), "Every active-deck card should have a notebook label."
non_energy_over_4 = [card_id for card_id, count in counts.items() if card_id != 6 and count > 4]
assert not non_energy_over_4, non_energy_over_4
assert basic_count >= 1, "active deck has no Basic Pokemon"

rows = []
for card_id, count in sorted(counts.items(), key=lambda kv: (CARD_LABELS[kv[0]][1], CARD_LABELS[kv[0]][0])):
    name, card_type, subtype = CARD_LABELS[card_id]
    rows.append({"card_id": card_id, "name": name, "count": count, "type": card_type, "subtype": subtype})

if pd is not None:
    deck_df = pd.DataFrame(rows)
    display(deck_df)
    display(deck_df.groupby("type", as_index=False)["count"].sum())
else:
    for row in rows:
        print(row)

BUILD_DIR.mkdir(exist_ok=True)
(BUILD_DIR / "deck.csv").write_text("\n".join(map(str, DECK)) + "\n")

print(f"Active deck variant: {DECK_VARIANT}")
print("deck length:", len(DECK))
print("unique cards:", len(counts))
print("basic pokemon count:", basic_count)
print("P(no Basic in opening 7):", round(prob_no_basic_opening_7, 4))
print("deck.csv:", BUILD_DIR / "deck.csv")

,card_id,name,count,type,subtype
0,6,Basic Fighting Energy,13,Energy,Basic Energy
1,674,Hariyama,2,Pokemon,Stage 1
2,675,Lunatone,2,Pokemon,Basic
3,673,Makuhita,2,Pokemon,Basic
4,678,Mega Lucario ex,4,Pokemon,Mega ex
5,677,Riolu,4,Pokemon,Basic
6,676,Solrock,3,Pokemon,Basic
7,1182,Boss's Orders,3,Trainer,Supporter
8,1192,Carmine,4,Trainer,Supporter
9,1102,Dusk Ball,4,Trainer,Item


,type,count
0,Energy,13
1,Pokemon,17
2,Trainer,30


Active deck variant: tempo
deck length: 60
unique cards: 16
basic pokemon count: 11
P(no Basic in opening 7): 0.2224
deck.csv: /kaggle/working/phase1_lucario_v2_build/deck.csv


## Decision Stack

The policy is organized as a stack of increasingly specific decisions:

1. parse the legal `select.option` list,
2. build a turn-local `AttackPlan`,
3. score attackers and targets by prize value, energy, stage, tools, and remaining HP,
4. rank legal actions by context,
5. apply a legal-selection filter,
6. optionally use forward search to re-rank high-leverage main-phase choices.

This structure keeps the rule policy transparent while leaving room for controlled search and ablation.

In [3]:
MAIN_PRELUDE = 'from __future__ import annotations\n\nimport os\nfrom collections import defaultdict\nimport time\nimport random\n\ntry:\n    from cg.api import search_begin, search_step, search_release\n    _SEARCH_AVAILABLE = True\nexcept Exception:\n    try:\n        from cg.api import search_begin, search_step, search_end as search_release\n        _SEARCH_AVAILABLE = True\n    except Exception:\n        _SEARCH_AVAILABLE = False\n\nfrom cg.api import (\n    AreaType,\n    Card,\n    CardType,\n    EnergyType,\n    Observation,\n    OptionType,\n    Pokemon,\n    SelectContext,\n    all_card_data,\n    to_observation_class,\n)\n\n\nclass C:\n    MAKUHITA = 673\n    HARIYAMA = 674\n    LUNATONE = 675\n    SOLROCK = 676\n    RIOLU = 677\n    MEGA_LUCARIO_EX = 678\n\n    BASIC_FIGHTING_ENERGY = 6\n    DUSK_BALL = 1102\n    SWITCH = 1123\n    PREMIUM_POWER_PRO = 1141\n    FIGHTING_GONG = 1142\n    POKE_PAD = 1152\n    HERO_CAPE = 1159\n    BOSS_ORDERS = 1182\n    CARMINE = 1192\n    LILLIE_DETERMINATION = 1227\n    GRAVITY_MOUNTAIN = 1252\n\n    LUMIOSE_CITY = 1267\n    LILLIES_PEARL = 1172\n    LEGACY_ENERGY = 12\n\n\nMEGA_BRAVE = 983\nLOW_DECK_COUNT = 8\n\nEMBEDDED_DECK = [\n    673, 673, 674, 674, 675, 675, 676, 676,\n    676, 677, 677, 677, 678, 678, 678, 678,\n    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,\n    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,\n    1152, 1152, 1159, 1182, 1182, 1192, 1192, 1192,\n    1192, 1227, 1227, 1227, 1227, 1252, 1252, 6,\n    6, 6, 6, 6, 6, 6, 6, 6,\n    6, 6, 6, 6,\n]\n\n\ndef _resolve_deck_path() -> str:\n    base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()\n    candidates = [\n        os.path.join(base_dir, "deck.csv"),\n        "deck.csv",\n        "/kaggle_simulations/agent/deck.csv",\n    ]\n    for path in candidates:\n        if os.path.exists(path):\n            return path\n    raise FileNotFoundError("deck.csv not found in: " + ", ".join(candidates))\n\n\ndef _load_deck() -> list[int]:\n    try:\n        deck_path = _resolve_deck_path()\n        with open(deck_path, "r", encoding="utf-8") as f:\n            deck = [int(line) for line in f.read().splitlines() if line.strip()]\n        if len(deck) == 60:\n            return deck\n    except Exception:\n        pass\n    return EMBEDDED_DECK.copy()\n\n\nmy_deck = _load_deck()\nif len(my_deck) != 60:\n    raise ValueError(f"deck must contain 60 card ids, got {len(my_deck)}")\n\n\nall_card = all_card_data()\ncard_table = {card.cardId: card for card in all_card}\n\n\nclass AttackPlan:\n    def __init__(\n        self,\n        attacker: int = -1,\n        target: int = -1,\n        attack_index: int = -1,\n        remain_hp: int = -1,\n        needs_energy: bool = False,\n    ):\n        self.attacker = attacker\n        self.target = target\n        self.attack_index = attack_index\n        self.remain_hp = remain_hp\n        self.needs_energy = needs_energy\n\n\nplan = AttackPlan()\npre_turn = -1\nability_used = False\n\n# Optional search hook. The Phase 1 notebook keeps this disabled; it exists so later\n# experiments can be connected without changing the submission contract.\nUSE_SEARCH = False'

In [4]:
MAIN_DIAGNOSTICS = '# --- Phase-1 diagnostics: distinguish "policy ran" from "fell back". ---------\n# A submission that never crashes can still be silently broken: if the policy\n# touches an SDK attribute that does not exist on the live engine, the exception\n# is caught and we return _legal_fallback (the first minCount indices). "0 errors"\n# then hides a near-random agent. _DIAG makes the fallback rate and the actual\n# exception messages observable so the validation gate can catch this.\n_DIAG = {"decisions": 0, "policy_ok": 0, "policy_fallback": 0,\n         "obs_fallback": 0, "deck_returns": 0, "errors": {},\n         "policy_errors": {}, "search_errors": {},\n         "search_used": 0, "search_failed": 0, "search_disabled": 0,\n         "chosen_types": {}, "attack_ids_chosen": {},\n         "attack_opts_by_active": {}, "mega_brave_present": 0}\n\n\ndef _diag_record_error(exc):\n    key = type(exc).__name__ + ": " + str(exc)[:160]\n    _DIAG["errors"][key] = _DIAG["errors"].get(key, 0) + 1\n    _DIAG["policy_errors"][key] = _DIAG["policy_errors"].get(key, 0) + 1\n\n\ndef _diag_record_search_error(exc):\n    key = type(exc).__name__ + ": " + str(exc)[:160]\n    _DIAG["errors"][key] = _DIAG["errors"].get(key, 0) + 1\n    _DIAG["search_errors"][key] = _DIAG["search_errors"].get(key, 0) + 1\n\n\ndef diag_reset():\n    _DIAG.update({"decisions": 0, "policy_ok": 0, "policy_fallback": 0,\n                  "obs_fallback": 0, "deck_returns": 0, "errors": {},\n                  "policy_errors": {}, "search_errors": {},\n                  "search_used": 0, "search_failed": 0, "search_disabled": 0,\n                  "chosen_types": {}, "attack_ids_chosen": {},\n                  "attack_opts_by_active": {}, "mega_brave_present": 0})\n\n\ndef diag_snapshot():\n    snap = {}\n    for k, v in _DIAG.items():\n        if k == "attack_opts_by_active":\n            snap[k] = {kk: sorted(vv) for kk, vv in v.items()}\n        elif isinstance(v, dict):\n            snap[k] = dict(v)\n        else:\n            snap[k] = v\n    dec = max(1, snap.get("decisions", 0))\n    snap["fallback_rate"] = (snap.get("policy_fallback", 0) + snap.get("obs_fallback", 0)) / dec\n    sd = snap.get("search_used", 0) + snap.get("search_failed", 0)\n    snap["search_fail_rate"] = (snap.get("search_failed", 0) / sd) if sd else 0.0\n    return snap\n\n\ndef _diag_observe(obs):\n    """Record, on the REAL obs, which attacks are legal (reveals true attackIds)."""\n    try:\n        sel = obs.select\n        st = obs.current\n        if sel is None or st is None:\n            return\n        me = st.players[st.yourIndex]\n        active_id = me.active[0].id if (me.active and me.active[0] is not None) else -1\n        for opt in sel.option:\n            if opt.type == OptionType.ATTACK:\n                aid = getattr(opt, "attackId", None)\n                if aid is not None:\n                    _DIAG["attack_opts_by_active"].setdefault(active_id, set()).add(aid)\n                    if aid == MEGA_BRAVE:\n                        _DIAG["mega_brave_present"] += 1\n    except Exception:\n        pass\n\n\ndef _diag_observe_choice(obs, selection):\n    """Record which option TYPES (and attackIds) actually get chosen."""\n    try:\n        sel = obs.select\n        if sel is None or not selection:\n            return\n        for i in selection:\n            if 0 <= i < len(sel.option):\n                opt = sel.option[i]\n                key = str(opt.type)\n                _DIAG["chosen_types"][key] = _DIAG["chosen_types"].get(key, 0) + 1\n                if opt.type == OptionType.ATTACK:\n                    aid = getattr(opt, "attackId", None)\n                    if aid is not None:\n                        _DIAG["attack_ids_chosen"][aid] = _DIAG["attack_ids_chosen"].get(aid, 0) + 1\n    except Exception:\n        pass'

## Forward Search Layer

The search layer is additive. Greedy Lucario v2 remains the fallback policy, while engine lookahead is used only when the runtime supports it and the current state exposes engine search input.

Key safeguards:

- search is bounded by node budget, wall-clock budget, and action cap,
- each root candidate starts from a fresh engine search state,
- simulated sub-decisions use the same greedy policy,
- repeated search failures automatically disable further search attempts,
- diagnostics track search usage, search failure rate, and fallback rate.

Greedy play is the stable default. Search remains a gated tactical layer for high-leverage main-phase positions until diagnostics prove that it is both reliable and stronger than greedy play.

In [5]:
MAIN_FORWARD_SEARCH = '# ---------------------------------------------------------------------------\n# Forward-search over the engine model. This uses the live SDK convention:\n# the observation dictionary carries search_begin_input, search_begin returns a\n# wrapper with {error, state}, search_step advances a searchId, and search_release\n# must be called for every started branch.\n# ---------------------------------------------------------------------------\nSEARCH_NODE_BUDGET = 120        # max forward-steps per decision\nSEARCH_TIME_BUDGET_S = 1.5      # wall-clock cap per decision (respect 10-min/game)\nSEARCH_ACTION_CAP = 20          # root MAIN candidates evaluated by search\nSEARCH_FAILURE_DISABLE_AFTER = 8\nSEARCH_MIN_ATTEMPTS_FOR_RATE = 10\nSEARCH_FAIL_RATE_DISABLE = 0.75\nSEARCH_OPPONENT_PLY = False     # opponent reply search is experimental, off by default\n\n\ndef _search_temporarily_disabled():\n    attempts = _DIAG.get("search_used", 0) + _DIAG.get("search_failed", 0)\n    failed = _DIAG.get("search_failed", 0)\n    used = _DIAG.get("search_used", 0)\n    if failed >= SEARCH_FAILURE_DISABLE_AFTER and used == 0:\n        return True\n    if attempts >= SEARCH_MIN_ATTEMPTS_FOR_RATE and failed / max(1, attempts) >= SEARCH_FAIL_RATE_DISABLE:\n        return True\n    return False\n\n\ndef _board_value(obs, my_index):\n    """Win-aligned leaf score used only by optional forward search."""\n    st = obs.current\n    res = getattr(st, "result", -1)\n    if res is not None and res >= 0:\n        if res == my_index:\n            return 1_000_000.0\n        if res == (1 - my_index):\n            return -1_000_000.0\n        return 0.0\n\n    me = st.players[my_index]\n    op = st.players[1 - my_index]\n    val = 10_000.0 * (len(op.prize) - len(me.prize))\n\n    my_active = me.active[0] if me.active else None\n    op_active = op.active[0] if op.active else None\n    if op_active is not None:\n        val += 2.0 * _damage_on(op_active)\n        val += 800.0 * prize_count(op_active)\n    if my_active is not None:\n        val -= 1.5 * _damage_on(my_active)\n        val -= 600.0 * prize_count(my_active)\n\n    ready_attackers = 0\n    for pkm in (me.active + me.bench):\n        if pkm is None:\n            continue\n        energy_count = len(getattr(pkm, "energies", []))\n        if pkm.id == C.MEGA_LUCARIO_EX and energy_count >= 2:\n            ready_attackers += 1\n        elif pkm.id == C.HARIYAMA and energy_count >= 3:\n            ready_attackers += 1\n        elif pkm.id == C.SOLROCK and energy_count >= 1:\n            ready_attackers += 1\n        val += 30.0 * energy_count\n\n    val += 300.0 * ready_attackers\n    val += 20.0 * getattr(me, "handCount", 0)\n    if getattr(me, "deckCount", 0) <= 4:\n        val -= 500.0 * (5 - getattr(me, "deckCount", 0))\n    return val\n\n\ndef _should_search(obs, ranked, scores):\n    sel = obs.select\n    if sel is None or obs.current is None:\n        return False\n    if sel.context != SelectContext.MAIN:\n        return False\n    if len(sel.option) <= 1:\n        return False\n    option_types = {opt.type for opt in sel.option}\n    tactical_types = {\n        OptionType.ATTACK,\n        OptionType.ATTACH,\n        OptionType.EVOLVE,\n        OptionType.RETREAT,\n        OptionType.PLAY,\n    }\n    if not (option_types & tactical_types):\n        return False\n    if plan.remain_hp <= 0:\n        return True\n    if plan.target >= 1:\n        return True\n    if plan.attacker > 0:\n        return True\n    if plan.needs_energy:\n        return True\n    return False\n\n\ndef _result_state(result):\n    if result is None:\n        return None\n    if isinstance(result, dict):\n        error = result.get("error", 0)\n        state = result.get("state")\n    else:\n        error = getattr(result, "error", 0)\n        state = getattr(result, "state", None)\n    if error not in (0, None):\n        return None\n    if state is not None:\n        return state\n    if hasattr(result, "observation") or hasattr(result, "searchId"):\n        return result\n    return None\n\n\ndef _state_id(state):\n    if isinstance(state, dict):\n        return state.get("searchId")\n    return getattr(state, "searchId", None)\n\n\ndef _state_obs(state):\n    if isinstance(state, dict):\n        return state.get("observation")\n    return getattr(state, "observation", None)\n\n\ndef _release_search(sid):\n    try:\n        search_release(sid)\n    except TypeError:\n        try:\n            search_release()\n        except Exception:\n            pass\n    except Exception:\n        pass\n\n\ndef _search_step_state(sid, selection):\n    if sid is None:\n        return None\n    return _result_state(search_step(sid, selection))\n\n\ndef _legal_sim_selection(obs, preferred_first=None):\n    try:\n        policy = LucarioPolicy(obs)\n        ranked, scores = policy.rank()\n        if preferred_first is not None:\n            ranked = [preferred_first] + [i for i in ranked if i != preferred_first]\n        selection = normalize_selection(ranked, scores, obs.select)\n        return selection if selection else _legal_fallback(obs.select)\n    except Exception:\n        return _legal_fallback(obs.select)\n\n\ndef _rollout_search_candidate(sbi, first_idx, my_index, t0):\n    sid = None\n    steps = 0\n    cur = None\n    try:\n        state = _result_state(search_begin(sbi))\n        if state is None:\n            return None, steps\n        sid = _state_id(state)\n        cur = _state_obs(state)\n        if cur is None or cur.select is None:\n            return None, steps\n\n        selection = _legal_sim_selection(cur, preferred_first=first_idx)\n        while steps < SEARCH_NODE_BUDGET and (time.time() - t0) <= SEARCH_TIME_BUDGET_S:\n            state = _search_step_state(sid, selection)\n            if state is None:\n                return None, steps\n            steps += 1\n            cur = _state_obs(state)\n            if cur is None or cur.current is None:\n                break\n            result = getattr(cur.current, "result", -1)\n            if result is not None and result >= 0:\n                break\n            if cur.current.yourIndex != my_index:\n                break\n            if cur.select is None or len(cur.select.option) == 0:\n                break\n            selection = _legal_sim_selection(cur)\n            if not selection:\n                break\n            if cur.select.context == SelectContext.MAIN:\n                try:\n                    if any(cur.select.option[i].type == OptionType.END for i in selection if 0 <= i < len(cur.select.option)):\n                        state = _search_step_state(sid, selection)\n                        if state is not None:\n                            cur = _state_obs(state) or cur\n                            steps += 1\n                        break\n                except Exception:\n                    pass\n        if cur is None:\n            return None, steps\n        return _board_value(cur, my_index), steps\n    finally:\n        if sid is not None:\n            _release_search(sid)\n\n\ndef search_plan(obs_dict, obs, ranked, scores):\n    if not _SEARCH_AVAILABLE:\n        return None\n    if _search_temporarily_disabled():\n        _DIAG["search_disabled"] += 1\n        return None\n    try:\n        if obs.select is None or obs.current is None:\n            return None\n        if obs.select.context != SelectContext.MAIN:\n            return None\n        if obs.select.maxCount == 0 or len(obs.select.option) <= 1:\n            return None\n        if not _should_search(obs, ranked, scores):\n            return None\n        sbi = getattr(obs, "search_begin_input", None)\n        if sbi is None and isinstance(obs_dict, dict):\n            sbi = obs_dict.get("search_begin_input")\n        if sbi is None:\n            return None\n\n        my_index = obs.current.yourIndex\n        candidates = []\n        for idx in ranked:\n            if not (0 <= idx < len(obs.select.option)) or idx in candidates:\n                continue\n            score = scores[idx] if idx < len(scores) else 0\n            if score <= 0 and obs.select.minCount == 0:\n                continue\n            candidates.append(idx)\n            if len(candidates) >= SEARCH_ACTION_CAP:\n                break\n        if not candidates:\n            return None\n\n        t0 = time.time()\n        best_idx, best_val = None, None\n        total_steps = 0\n        for first_idx in candidates:\n            if total_steps >= SEARCH_NODE_BUDGET or (time.time() - t0) > SEARCH_TIME_BUDGET_S:\n                break\n            value, used_steps = _rollout_search_candidate(sbi, first_idx, my_index, t0)\n            total_steps += max(1, used_steps)\n            if value is None:\n                continue\n            if best_val is None or value > best_val:\n                best_idx, best_val = first_idx, value\n        if best_idx is None:\n            _DIAG["search_failed"] += 1\n            return None\n        _DIAG["search_used"] += 1\n        return [best_idx] + [i for i in ranked if i != best_idx]\n    except Exception as exc:\n        _diag_record_search_error(exc)\n        _DIAG["search_failed"] += 1\n        return None\n'

## Context-Aware Score Floor

The legal action space is heterogeneous: setup, bench placement, card search, discard, damage counters, retreat, attach, evolve, ability, and attack choices all share the same index-return interface.

The important rule is that unmodeled optional choices should not be selected just because they exist. A score of zero is treated as neutral/unknown, not positive.

The context-aware floor therefore uses:

- `score > 0` for optional choices,
- `minCount` compliance when a choice is mandatory,
- explicit scoring for setup bench, to-bench, discard, and damage-counter contexts,
- fallback only when policy ranking itself fails.

In [6]:
MAIN_SELECTION_HELPERS = 'def normalize_selection(ranked, scores, select):\n    n = len(select.option)\n    minc = max(0, min(select.minCount, n))\n    maxc = max(minc, min(select.maxCount, n))\n\n    out = []\n    seen = set()\n    for i in ranked:\n        if not (0 <= i < n) or i in seen:\n            continue\n        score = scores[i] if i < len(scores) else 0\n        # Score 0 usually means an unmodelled neutral option. For optional\n        # selections, do not take it unless minCount forces a choice.\n        if score > 0 or len(out) < minc:\n            out.append(i)\n            seen.add(i)\n        if len(out) >= maxc:\n            break\n\n    for i in range(n):\n        if len(out) >= minc:\n            break\n        if i not in seen:\n            out.append(i)\n            seen.add(i)\n    return out\n\n\ndef _legal_fallback(select):\n    try:\n        n = len(select.option)\n        k = min(max(0, select.minCount), n)\n        return list(range(k))\n    except Exception:\n        return []\n\n\ndef _legal_fallback_from_dict(obs_dict):\n    try:\n        sel = obs_dict.get("select") or {}\n        opts = sel.get("option") or []\n        minc = sel.get("minCount", 0)\n        n = len(opts)\n        k = min(max(0, minc), n)\n        return list(range(k))\n    except Exception:\n        return []\n\ndef _safe_get(seq, index: int):\n    try:\n        if seq is None or index is None or index < 0 or index >= len(seq):\n            return None\n        return seq[index]\n    except Exception:\n        return None'

In [7]:
MAIN_CARD_HELPERS = 'def _ctx_is(context, *names) -> bool:\n    for name in names:\n        value = getattr(SelectContext, name, None)\n        if value is not None and context == value:\n            return True\n    return False\n\n\ndef _pokemon_max_hp(pokemon: Pokemon) -> int:\n    max_hp = getattr(pokemon, "maxHp", None)\n    if max_hp is not None:\n        return max_hp\n    data = card_table.get(getattr(pokemon, "id", None))\n    return getattr(data, "hp", getattr(pokemon, "hp", 0))\n\n\ndef _damage_on(pokemon: Pokemon) -> int:\n    return max(0, _pokemon_max_hp(pokemon) - getattr(pokemon, "hp", 0))\n\n\ndef get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:\n    try:\n        player = obs.current.players[player_index]\n        match area:\n            case AreaType.DECK:\n                return _safe_get(getattr(obs.select, "deck", None), index)\n            case AreaType.HAND:\n                return _safe_get(getattr(player, "hand", None), index)\n            case AreaType.DISCARD:\n                return _safe_get(getattr(player, "discard", None), index)\n            case AreaType.ACTIVE:\n                return _safe_get(getattr(player, "active", None), index)\n            case AreaType.BENCH:\n                return _safe_get(getattr(player, "bench", None), index)\n            case AreaType.PRIZE:\n                return _safe_get(getattr(player, "prize", None), index)\n            case AreaType.STADIUM:\n                return _safe_get(getattr(obs.current, "stadium", None), index)\n            case AreaType.LOOKING:\n                return _safe_get(getattr(obs.current, "looking", None), index)\n            case _:\n                return None\n    except Exception:\n        return None\n\n\ndef prize_count(pokemon: Pokemon) -> int:\n    data = card_table.get(pokemon.id)\n    if data is None:\n        return 1\n    count = 3 if data.megaEx else 2 if data.ex else 1\n    for card in pokemon.energyCards:\n        if card.id == C.LEGACY_ENERGY:\n            count -= 1\n    for card in pokemon.tools:\n        if card.id == C.LILLIES_PEARL and "Lillie" in data.name:\n            count -= 1\n    return max(0, count)\n\n\ndef target_score(pokemon: Pokemon) -> int:\n    data = card_table.get(pokemon.id)\n    if data is None:\n        return prize_count(pokemon) * 1000 + getattr(pokemon, "hp", 0) + _damage_on(pokemon) * 2\n    score = prize_count(pokemon) * 1000\n    score += len(pokemon.energies) * 150\n    score += len(pokemon.tools) * 100\n    if data.stage2:\n        score += 250\n    elif data.stage1:\n        score += 130\n\n    if pokemon.id in {144, 322, 323, 337}:  # low-value support Pokemon\n        score -= 200\n    if pokemon.id == 112 and len(pokemon.energies) >= 1:  # Munkidori\n        score += 300\n    score += pokemon.hp\n    return score'


## Lucario Policy Body

The rule policy keeps the public Lucario v2 structure and adds stronger safety boundaries.

Policy components:

- low-deck draw suppression,
- prize-aware target scoring,
- attack planning across attach/evolve/switch/Boss/attack decisions,
- setup and bench scoring for the Lucario, Solrock/Lunatone, and Hariyama lines,
- max-HP-aware board evaluation and evolution timing guards,
- explicit scoring for heal, effect-target, field-placement, mulligan, and evolution sub-contexts,
- chosen-action side effects based on the actual selected indices.

This keeps optional action quality high without relying on neural inference.

In [8]:
MAIN_POLICY = 'class LucarioPolicy:\n    def __init__(self, obs: Observation):\n        self.obs = obs\n        self.state = obs.current\n        self.select = obs.select\n        self.context = self.select.context\n        self.my_index = self.state.yourIndex\n        self.op_index = 1 - self.my_index\n        self.me = self.state.players[self.my_index]\n        self.opponent = self.state.players[self.op_index]\n        self.my_prizes_left = len(self.me.prize)\n\n        self.field_counts = defaultdict(int)\n        self.hand_counts = defaultdict(int)\n        self.discard_counts = defaultdict(int)\n        self.has_ready_lucario_line = False\n        self.has_ready_hariyama_line = False\n        self.can_switch = False\n        self.can_gust = False\n        self.can_attack = False\n        self.can_use_mega_brave = False\n        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0\n\n        self._count_cards()\n        self._scan_main_options()\n\n    def rank(self) -> tuple[list[int], list[float]]:\n        if not self.select.option or self.select.maxCount == 0:\n            return [], []\n\n        if self.context == SelectContext.MAIN:\n            self._plan_attack()\n\n        scores = [self._score_option(option) for option in self.select.option]\n        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]\n        return ranked, scores\n\n    def choose(self) -> list[int]:\n        ranked, scores = self.rank()\n        selection = normalize_selection(ranked, scores, self.select)\n        self.remember_chosen_side_effects(selection)\n        return selection\n    def _count_cards(self) -> None:\n        for pokemon in self.me.active + self.me.bench:\n            if pokemon is None:\n                continue\n            self.field_counts[pokemon.id] += 1\n            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3:\n                self.has_ready_hariyama_line = True\n            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2:\n                self.has_ready_lucario_line = True\n\n        for card in self.me.hand:\n            self.hand_counts[card.id] += 1\n        for card in self.me.discard:\n            self.discard_counts[card.id] += 1\n\n    def _scan_main_options(self) -> None:\n        if self.context != SelectContext.MAIN:\n            return\n        for option in self.select.option:\n            if option.type == OptionType.PLAY:\n                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n                if card is None:\n                    continue\n                if card.id == C.SWITCH:\n                    self.can_switch = True\n                elif card.id == C.BOSS_ORDERS:\n                    self.can_gust = True\n            elif option.type == OptionType.EVOLVE:\n                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n                if card is None:\n                    continue\n                if card.id == C.HARIYAMA:\n                    self.can_gust = True\n            elif option.type == OptionType.RETREAT:\n                self.can_switch = True\n            elif option.type == OptionType.ATTACK:\n                self.can_attack = True\n                if option.attackId == MEGA_BRAVE:\n                    self.can_use_mega_brave = True\n\n    def _my_board(self) -> list[Pokemon | None]:\n        return self.me.active + self.me.bench\n\n    def _opponent_board(self) -> list[Pokemon | None]:\n        return self.opponent.active + self.opponent.bench\n\n    def _can_evolve_board_index(self, board_index: int) -> bool:\n        for option in self.select.option:\n            if option.type != OptionType.EVOLVE:\n                continue\n            target_index = option.inPlayIndex\n            if option.inPlayArea == AreaType.BENCH:\n                target_index += 1\n            if target_index == board_index:\n                return True\n        return False\n\n    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:\n        energy_required = 0\n        base_damage = 0\n        base_score = 0\n\n        if pokemon.id == C.MEGA_LUCARIO_EX:\n            if attack_index == 0:\n                energy_required = 1\n                base_damage = 130\n                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])\n            else:\n                energy_required = 2\n                base_damage = 270\n            if self.my_prizes_left in {2, 3}:\n                base_score -= 500\n        elif attack_index == 1:\n            return None\n        elif pokemon.id == C.HARIYAMA:\n            energy_required = 3\n            base_damage = 210\n        elif pokemon.id == C.MAKUHITA:\n            return None\n        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1:\n            energy_required = 1\n            base_damage = 70\n\n        if base_damage <= 0:\n            return None\n        return energy_required, base_damage, base_score\n\n    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):\n        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index):\n            return 3, 210, -100\n        return self._base_attack(pokemon, attack_index)\n\n    def _plan_attack(self) -> None:\n        global plan\n        best_score = -1\n        plan = AttackPlan()\n\n        if self.state.turn < 2:\n            return\n\n        for attacker_index, my_pokemon in enumerate(self._my_board()):\n            if my_pokemon is None:\n                continue\n            if attacker_index != 0 and not self.can_switch:\n                break\n\n            for attack_index in range(2):\n                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)\n                if attack is None:\n                    continue\n                energy_required, base_damage, base_score = attack\n\n                energy_count = len(my_pokemon.energies)\n                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave:\n                    break\n\n                needs_energy = False\n                if energy_count < energy_required:\n                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:\n                        energy_count += 1\n                        needs_energy = energy_count >= energy_required\n                    if not needs_energy:\n                        continue\n\n                for target_index, op_pokemon in enumerate(self._opponent_board()):\n                    if op_pokemon is None:\n                        continue\n                    if target_index != 0 and not self.can_gust:\n                        break\n\n                    damage = base_damage\n                    op_data = card_table.get(op_pokemon.id)\n                    if op_data is None:\n                        continue\n                    if op_data.weakness == EnergyType.FIGHTING:\n                        damage *= 2\n                    elif op_data.resistance == EnergyType.FIGHTING:\n                        damage -= 30\n\n                    score = target_score(op_pokemon)\n                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0\n                    if prize == 0:\n                        score *= damage / op_pokemon.hp\n                    if len(self.opponent.prize) <= prize:\n                        score = 50000\n\n                    score += base_score\n                    score += 220 if attacker_index == 0 else 0\n                    score += 300 if target_index == 0 else 0\n                    score += energy_count\n\n                    if score > best_score:\n                        best_score = score\n                        plan = AttackPlan(\n                            attacker=attacker_index,\n                            target=target_index,\n                            attack_index=attack_index,\n                            remain_hp=op_pokemon.hp - damage,\n                            needs_energy=needs_energy,\n                        )\n\n    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:\n        energy_count = len(pokemon.energies)\n        score = 8000 + (10 if active else 0)\n\n        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:\n            score += 1 if pokemon.id == C.HARIYAMA else 0\n            score += 100 if energy_count < 3 else 0\n            score -= 50 if self.has_ready_hariyama_line else 0\n        elif pokemon.id == C.LUNATONE:\n            score -= 100\n        elif pokemon.id == C.SOLROCK:\n            score += 20 if energy_count < 1 else -100\n        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:\n            score += 1 if pokemon.id == C.MEGA_LUCARIO_EX else 0\n            score += 100 if energy_count < 2 else 0\n            score -= 50 if self.has_ready_lucario_line else 0\n        return score\n\n    def _score_option(self, option) -> float:\n        if option.type == OptionType.NUMBER:\n            return option.number\n        if option.type == OptionType.YES:\n            return 100 if self.context == SelectContext.IS_FIRST else 1\n        if option.type == OptionType.NO:\n            return 0\n        if option.type == OptionType.CARD:\n            return self._score_card_choice(option)\n        if option.type == OptionType.PLAY:\n            return self._score_play(option)\n        if option.type == OptionType.ATTACH:\n            return self._score_attach(option)\n        if option.type == OptionType.EVOLVE:\n            return self._score_evolve(option)\n        if option.type == OptionType.ABILITY:\n            return self._score_ability(option)\n        if option.type == OptionType.RETREAT:\n            return 2000 if plan.attacker >= 1 else -1\n        if option.type == OptionType.ATTACK:\n            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000\n        return 0\n\n    def _score_card_choice(self, option) -> float:\n        card = get_card(self.obs, option.area, option.index, option.playerIndex)\n        if card is None:\n            return 0\n\n        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:\n            return self._score_active_choice(option, card)\n        if self.context == SelectContext.SETUP_ACTIVE_POKEMON:\n            return self._score_setup_active(card)\n        if self.context == SelectContext.SETUP_BENCH_POKEMON:\n            return self._score_setup_bench(card)\n        if self.context == SelectContext.TO_HAND:\n            return self._score_to_hand(card)\n        if self.context == SelectContext.TO_BENCH:\n            return self._score_to_bench(card)\n        if _ctx_is(self.context, "TO_FIELD"):\n            return self._score_to_field(option, card)\n        if self.context == SelectContext.DISCARD:\n            return self._score_discard(card)\n        if self.context in {SelectContext.DAMAGE_COUNTER, SelectContext.DAMAGE_COUNTER_ANY}:\n            return self._score_damage_counter(option, card)\n        if _ctx_is(self.context, "DAMAGE", "EFFECT_TARGET"):\n            return self._score_effect_target(option, card)\n        if _ctx_is(self.context, "HEAL", "REMOVE_DAMAGE_COUNTER"):\n            return self._score_heal_target(option, card)\n        if _ctx_is(self.context, "EVOLVES_FROM", "EVOLVES_TO"):\n            return self._score_evolution_context(option, card)\n        if _ctx_is(self.context, "MULLIGAN"):\n            return self._score_mulligan(card)\n        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon):\n            return self._energy_target_score(card, option.area == AreaType.ACTIVE)\n        return 0\n\n    def _score_active_choice(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return 0\n\n        if option.playerIndex != self.my_index:\n            return 100 if option.index == plan.target - 1 else 0\n\n        score = len(card.energies) * 2\n        if option.index == plan.attacker - 1:\n            score += 100\n        if card.id == C.MEGA_LUCARIO_EX:\n            score += 8 if self.my_prizes_left in {2, 3} else 20\n        elif card.id == C.HARIYAMA and len(card.energies) >= 2:\n            score += 15\n        elif card.id == C.MAKUHITA and len(card.energies) >= 2:\n            score += 10\n        elif card.id == C.SOLROCK:\n            score += 5\n        elif card.id == C.RIOLU:\n            score += 4\n        return score\n\n    def _score_setup_active(self, card: Pokemon | Card) -> int:\n        if card is None:\n            return 0\n        if card.id == C.SOLROCK:\n            return 2 if self.state.firstPlayer == self.my_index else 4\n        if card.id == C.RIOLU:\n            return 3\n        if card.id == C.MAKUHITA:\n            return 1\n        return 0\n\n    def _score_setup_bench(self, card: Pokemon | Card) -> int:\n        if card is None:\n            return 0\n        if card.id == C.RIOLU:\n            return 120 - 25 * self.field_counts[C.RIOLU]\n        if card.id == C.SOLROCK:\n            return 90 if self.field_counts[C.SOLROCK] == 0 else -1\n        if card.id == C.LUNATONE:\n            return 80 if self.field_counts[C.LUNATONE] == 0 else -1\n        if card.id == C.MAKUHITA:\n            return 65 if self.field_counts[C.MAKUHITA] == 0 else 10\n        return 0\n\n    def _score_to_bench(self, card: Pokemon | Card) -> float:\n        if card is None:\n            return 0\n        data = card_table.get(card.id)\n        if data is None or data.cardType != CardType.POKEMON:\n            return 0\n        return self._score_setup_bench(card)\n\n    def _score_discard(self, card: Pokemon | Card) -> float:\n        if card is None:\n            return 0\n        cid = card.id\n        # Positive means "safe/desirable to discard". Required discard contexts will\n        # still pick the highest values; optional discard contexts will skip <= 0.\n        if cid == C.BASIC_FIGHTING_ENERGY:\n            score = 45 if self.hand_counts[cid] >= 2 else 5\n            if plan.needs_energy and not self.state.energyAttached:\n                score -= 200\n            return score\n        if self.hand_counts[cid] >= 2:\n            return 70\n        if cid in {C.LUNATONE, C.SOLROCK} and self.field_counts[cid] >= 1:\n            return 55\n        if cid == C.GRAVITY_MOUNTAIN and self.stadium_id == C.GRAVITY_MOUNTAIN:\n            return 50\n        if cid in {C.CARMINE, C.LILLIE_DETERMINATION} and self.state.supporterPlayed:\n            return 30\n        if cid == C.MEGA_LUCARIO_EX and self.field_counts[C.RIOLU] == 0:\n            return -80\n        if cid == C.HARIYAMA and self.field_counts[C.MAKUHITA] == 0:\n            return -50\n        if cid in {C.RIOLU, C.MAKUHITA, C.BOSS_ORDERS, C.HERO_CAPE}:\n            return -40\n        return 0\n\n    def _is_opponent_option(self, option) -> bool:\n        return getattr(option, "playerIndex", self.my_index) == self.op_index\n\n    def _score_damage_counter(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return 0\n        if self._is_opponent_option(option):\n            return 10000 + prize_count(card) * 1000 - getattr(card, "hp", 0) + _damage_on(card) * 5\n        return -target_score(card)\n\n    def _score_effect_target(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return self._score_to_hand(card)\n        if self._is_opponent_option(option):\n            return 2000 + target_score(card) + _damage_on(card) * 8\n        score = 300 + len(card.energies) * 50 + len(card.tools) * 40 + _damage_on(card) * 8\n        if card.id == C.MEGA_LUCARIO_EX:\n            score += 250\n        elif card.id in {C.RIOLU, C.HARIYAMA}:\n            score += 120\n        elif card.id in {C.SOLROCK, C.LUNATONE, C.MAKUHITA}:\n            score += 70\n        return score\n\n    def _score_heal_target(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return 0\n        damage = _damage_on(card)\n        if self._is_opponent_option(option):\n            return -1000 - damage\n        score = damage * 20 + len(card.energies) * 30 + len(card.tools) * 25\n        if card.id == C.MEGA_LUCARIO_EX:\n            score += 300\n        elif card.id in {C.RIOLU, C.HARIYAMA}:\n            score += 120\n        return score if damage > 0 else max(0, score // 10)\n\n    def _score_evolution_context(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return 0\n        if self._is_opponent_option(option):\n            return target_score(card)\n        if card.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:\n            return 900 + len(card.energies) * 30\n        if card.id in {C.MAKUHITA, C.HARIYAMA}:\n            return 650 + len(card.energies) * 30\n        return 100 + len(card.energies) * 20\n\n    def _score_to_field(self, option, card: Pokemon | Card) -> float:\n        if isinstance(card, Pokemon):\n            return self._score_to_bench(card) + self._score_setup_active(card)\n        return self._score_to_hand(card)\n\n    def _score_mulligan(self, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon):\n            return 0\n        data = card_table.get(card.id)\n        if data is None:\n            return 0\n        is_basic = (\n            getattr(data, "basic", False)\n            or (\n                data.cardType == CardType.POKEMON\n                and not getattr(data, "stage1", False)\n                and not getattr(data, "stage2", False)\n                and not getattr(data, "megaEx", False)\n            )\n        )\n        if not is_basic:\n            return 0\n        return self._score_setup_active(card) + self._score_setup_bench(card)\n\n    def _score_to_hand(self, card: Pokemon | Card) -> float:\n        if card is None:\n            return 0\n        score = 200 - self.hand_counts[card.id] * 100\n        if card.id == C.MAKUHITA:\n            score += -10 if self.field_counts[card.id] >= 1 else 10\n        elif card.id == C.HARIYAMA:\n            score += 20 if self.field_counts[C.MAKUHITA] >= 1 else -20\n        elif card.id == C.LUNATONE:\n            score += -250 if self.field_counts[card.id] >= 1 else 60\n        elif card.id == C.SOLROCK:\n            score += -250 if self.field_counts[card.id] >= 1 else 50\n        elif card.id == C.RIOLU:\n            lucario_line = self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX]\n            score += -150 if lucario_line >= 2 else -3 if lucario_line >= 1 else 40\n        elif card.id == C.MEGA_LUCARIO_EX:\n            score += 40 if self.field_counts[C.RIOLU] >= 1 else -15\n        elif card.id == C.BASIC_FIGHTING_ENERGY:\n            score += 30 if not ability_used or not self.state.energyAttached else -1\n        return score\n\n    def _score_play(self, option) -> float:\n        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n        if card is None:\n            return 0\n        data = card_table.get(card.id)\n        if data is None:\n            return 0\n        if data.cardType == CardType.POKEMON:\n            return self._score_play_pokemon(card)\n        return self._score_play_trainer(card)\n\n    def _score_play_pokemon(self, card: Card) -> float:\n        score = 20000\n        if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1:\n            return -1\n        if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2:\n            return -1\n        return score\n\n    def _score_play_trainer(self, card: Card) -> float:\n        if card.id == C.SWITCH:\n            if plan.attacker > 0 and plan.remain_hp <= 0:\n                return 14000\n            return 6500 if plan.attacker > 0 else -1\n        if card.id == C.PREMIUM_POWER_PRO:\n            if self.state.supporterPlayed and plan.remain_hp <= 0:\n                return -1\n            if not self.can_attack:\n                can_bridge_draw = (\n                    not self.state.supporterPlayed\n                    and self.hand_counts[C.CARMINE] > 0\n                    and self.hand_counts[C.LILLIE_DETERMINATION] == 0\n                    and not self._low_deck()\n                )\n                return 3050 if can_bridge_draw else -1\n            return 5000\n        if card.id == C.BOSS_ORDERS:\n            if plan.target >= 1 and plan.remain_hp <= 0:\n                return 15000\n            return 4200 if plan.target >= 1 else -1\n        if card.id == C.CARMINE:\n            return -1 if self._low_deck() else 3000\n        if card.id == C.LILLIE_DETERMINATION:\n            return -1 if self._low_deck() else 3100\n        if card.id == C.GRAVITY_MOUNTAIN:\n            return self._score_gravity_mountain()\n        return 10000\n\n    def _score_gravity_mountain(self) -> float:\n        opponent_has_stage2 = any(\n            pokemon is not None and card_table.get(pokemon.id) is not None and card_table[pokemon.id].stage2\n            for pokemon in self._opponent_board()\n        )\n        if opponent_has_stage2:\n            return 3500\n        return 1200 if self.stadium_id else -1\n\n    def _low_deck(self) -> bool:\n        return self.me.deckCount <= LOW_DECK_COUNT\n\n    def _score_attach(self, option) -> float:\n        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)\n        if card is None or not isinstance(pokemon, Pokemon):\n            return 0\n\n        if card.id == C.HERO_CAPE:\n            score = 7000\n            if pokemon.id == C.RIOLU:\n                score += 100\n            elif pokemon.id == C.MEGA_LUCARIO_EX:\n                score += 200\n            return score\n\n        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)\n        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1\n        if board_index == plan.attacker and plan.needs_energy:\n            score += 200\n        return score\n\n    def _score_evolve(self, option) -> float:\n        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)\n        if not isinstance(pokemon, Pokemon):\n            return 0\n        if getattr(pokemon, "appearThisTurn", False):\n            return -1\n        if pokemon.id == C.MAKUHITA and plan.target == 0:\n            return -1\n        evolve_card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n        score = 9000 + len(pokemon.energies)\n        if evolve_card is not None:\n            if evolve_card.id == C.MEGA_LUCARIO_EX and pokemon.id == C.RIOLU:\n                score += 350\n            elif evolve_card.id == C.HARIYAMA and pokemon.id == C.MAKUHITA:\n                score += 180\n        return score\n\n    def _score_ability(self, option) -> float:\n        card = get_card(self.obs, option.area, option.index, self.my_index)\n        if card is None:\n            return 0\n        if card.id == C.LUMIOSE_CITY:\n            return 1\n        if card.id == C.LUNATONE and self._low_deck():\n            return -1\n        return 30000\n\n    def remember_chosen_side_effects(self, selection: list[int]) -> None:\n        global ability_used\n        if self.context != SelectContext.MAIN:\n            return\n        for idx in selection:\n            if not (0 <= idx < len(self.select.option)):\n                continue\n            option = self.select.option[idx]\n            if option.type != OptionType.ABILITY:\n                continue\n            card = get_card(self.obs, option.area, option.index, self.my_index)\n            if card is not None and card.id == C.LUNATONE:\n                ability_used = True'

In [9]:
MAIN_AGENT_ENTRYPOINT = 'def agent(obs_dict: dict) -> list[int]:\n    global pre_turn, ability_used, plan\n\n    # Deck-selection phase: the engine asks for the deck with select == None.\n    # Return the 60-card id list (different return semantics from a normal turn).\n    try:\n        select_is_none = isinstance(obs_dict, dict) and obs_dict.get("select") is None\n    except Exception:\n        select_is_none = False\n    if select_is_none:\n        _DIAG["deck_returns"] += 1\n        return my_deck\n\n    _DIAG["decisions"] += 1\n    try:\n        obs = to_observation_class(obs_dict)\n        if obs.select is None:\n            _DIAG["deck_returns"] += 1\n            _DIAG["decisions"] -= 1\n            return my_deck\n\n        if obs.current is not None and pre_turn != obs.current.turn:\n            pre_turn = obs.current.turn\n            ability_used = False\n            plan = AttackPlan()\n\n        try:\n            policy = LucarioPolicy(obs)\n            _diag_observe(obs)\n            ranked, scores = policy.rank()\n            planned = search_plan(obs_dict, obs, ranked, scores) if USE_SEARCH else None\n            ranked_use = ranked if planned is None else planned\n            selection = normalize_selection(ranked_use, scores, obs.select)\n            policy.remember_chosen_side_effects(selection)\n            _diag_observe_choice(obs, selection)\n            _DIAG["policy_ok"] += 1\n            return selection\n        except Exception as exc:\n            _diag_record_error(exc)\n            _DIAG["policy_fallback"] += 1\n            return _legal_fallback(obs.select)\n    except Exception as exc:\n        _diag_record_error(exc)\n        _DIAG["obs_fallback"] += 1\n        return _legal_fallback_from_dict(obs_dict if isinstance(obs_dict, dict) else {})'

In [10]:
MAIN_SECTIONS = [
    MAIN_PRELUDE,
    MAIN_DIAGNOSTICS,
    MAIN_FORWARD_SEARCH,
    MAIN_SELECTION_HELPERS,
    MAIN_CARD_HELPERS,
    MAIN_POLICY,
    MAIN_AGENT_ENTRYPOINT,
]
MAIN_CODE = "\n\n".join(section.rstrip() for section in MAIN_SECTIONS)

import re
MAIN_CODE, embedded_deck_replacements = re.subn(
    r"EMBEDDED_DECK = \[\n(?:    [0-9, ]+\n)+\]",
    "EMBEDDED_DECK = " + repr(DECK),
    MAIN_CODE,
    count=1,
)
assert embedded_deck_replacements == 1, "EMBEDDED_DECK replacement failed"
MAIN_CODE = MAIN_CODE.replace("USE_SEARCH = False", f"USE_SEARCH = {USE_SEARCH_IN_AGENT}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_NODE_BUDGET = 120", f"SEARCH_NODE_BUDGET = {SEARCH_NODE_BUDGET}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_TIME_BUDGET_S = 1.5", f"SEARCH_TIME_BUDGET_S = {SEARCH_TIME_BUDGET_S}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_ACTION_CAP = 20", f"SEARCH_ACTION_CAP = {SEARCH_ACTION_CAP}", 1)

BUILD_DIR.mkdir(exist_ok=True)
(BUILD_DIR / "main.py").write_text(MAIN_CODE)
print("wrote", BUILD_DIR / "main.py", "bytes=", len(MAIN_CODE.encode()),
      "| USE_SEARCH=", USE_SEARCH_IN_AGENT, "| node_budget=", SEARCH_NODE_BUDGET)

wrote /kaggle/working/phase1_lucario_v2_build/main.py bytes= 45372 | USE_SEARCH= False | node_budget= 120


In [11]:
main_text = (BUILD_DIR / "main.py").read_text()
compile(main_text, str(BUILD_DIR / "main.py"), "exec")
assert "def agent(obs_dict: dict)" in main_text
assert "import random" in main_text
assert "EMBEDDED_DECK = " + repr(DECK) in main_text
assert "def normalize_selection" in main_text
assert "USE_SEARCH = False" in main_text
assert "def search_plan" in main_text
assert "score > 0" in main_text
assert "_resolve_deck_path" in main_text
assert "remember_chosen_side_effects" in main_text
assert "search_begin_input" in main_text
assert "search_release(sid)" in main_text
assert "def _rollout_search_candidate" in main_text
assert "def _should_search" in main_text
assert "SearchState" not in main_text
assert "search_end()" not in main_text
assert "def _pokemon_max_hp" in main_text
assert "def _ctx_is" in main_text
assert "maxHp" in main_text
assert "appearThisTurn" in main_text
assert "def _score_effect_target" in main_text
assert "def _score_heal_target" in main_text
assert "def _score_evolution_context" in main_text
assert "def _score_mulligan" in main_text
assert "def _diag_record_search_error" in main_text
assert "search_errors" in main_text
assert "_score_setup_bench" in main_text
assert "submission.csv" not in main_text
print("main.py syntax and required hooks: OK")

main.py syntax and required hooks: OK


## Diagnostics And Evaluation

Evaluation separates policy quality from runtime accidents.

Useful signals:

- completed games,
- policy fallback rate,
- search failure rate,
- search failure gate when search is enabled,
- search disabled count,
- chosen action type distribution,
- attack IDs offered and chosen,
- mirror and random-opponent performance,
- sample-deck matchup behavior.

A policy change is only meaningful after fallback and runtime behavior are stable.

In [12]:
def iter_cg_candidates():
    explicit = [
        ROOT / "cg",
        Path("/kaggle/working/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle-simulation/cg"),
    ]
    patterns = [
        "/kaggle/input/**/cg-lib/cg",
        "/kaggle/input/**/cg",
    ]
    seen = set()
    for p in explicit:
        if p not in seen:
            seen.add(p)
            yield p
    for pattern in patterns:
        for match in glob.glob(pattern, recursive=True):
            p = Path(match)
            if p not in seen:
                seen.add(p)
                yield p


def copy_cg_package():
    for src in iter_cg_candidates():
        if src.exists() and src.is_dir() and (src / "api.py").exists():
            dst = BUILD_DIR / "cg"
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.pyo"))
            print("copied cg package from", src)
            return dst
    print("cg package not found in the current runtime.")
    return None

CG_DIR = copy_cg_package()




copied cg package from /kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg


In [13]:
# ============================================================================
# Shared helpers used by both the EDA cells and the validation gate.
# ============================================================================
def load_deck_ids():
    return [int(x.strip()) for x in (BUILD_DIR / "deck.csv").read_text().splitlines() if x.strip()]


def random_legal_agent_factory(deck):
    """A legal-but-random opponent: returns maxCount distinct legal option indices."""
    def random_legal_agent(obs_dict):
        sel = obs_dict.get("select")
        if sel is None:
            return deck
        options = sel.get("option") or []
        n = len(options)
        minc = max(0, min(sel.get("minCount", 0), n))
        maxc = max(minc, min(sel.get("maxCount", minc), n))
        return random.sample(range(n), maxc) if maxc else []
    return random_legal_agent


def import_generated_module():
    """Import the freshly-written main.py as a module so we can read its _DIAG."""
    import importlib.util
    if str(BUILD_DIR) not in sys.path:
        sys.path.insert(0, str(BUILD_DIR))
    spec = importlib.util.spec_from_file_location("phase1_generated_main", BUILD_DIR / "main.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


In [14]:
def run_optional_engine_eda():
    if CG_DIR is None:
        print("skip engine EDA: cg/ unavailable in this runtime")
        return None

    if str(BUILD_DIR) not in sys.path:
        sys.path.insert(0, str(BUILD_DIR))

    try:
        from cg.api import CardType, to_observation_class, all_card_data
        from cg.game import battle_start, battle_finish
    except Exception as e:
        print("skip engine EDA: cg import failed:", repr(e))
        return None

    cards = all_card_data()
    card_rows = []
    for card in cards:
        card_rows.append({
            "card_id": getattr(card, "cardId", None),
            "name": getattr(card, "name", None),
            "card_type": str(getattr(card, "cardType", None)).split(".")[-1],
            "hp": getattr(card, "hp", None),
            "ex": getattr(card, "ex", None),
            "megaEx": getattr(card, "megaEx", None),
        })

    if pd is not None:
        card_df = pd.DataFrame(card_rows)
        display(card_df.groupby("card_type", as_index=False).size())
        display(card_df[["hp"]].describe())
    else:
        print("total cards:", len(card_rows))

    deck = [int(x.strip()) for x in (BUILD_DIR / "deck.csv").read_text().splitlines() if x.strip()]
    obs_raw, start_data = battle_start(deck, deck)
    try:
        if obs_raw is None:
            print("battle_start failed:", start_data)
            return None

        obs = to_observation_class(obs_raw) if isinstance(obs_raw, dict) else obs_raw
        print("battle_start: OK")
        print("obs.current is None:", getattr(obs, "current", None) is None)
        print("obs.select is None:", getattr(obs, "select", None) is None)
        if getattr(obs, "select", None) is not None:
            print("select context:", getattr(obs.select, "context", None))
            print("select.option count:", len(obs.select.option))
            print("min/max:", obs.select.minCount, obs.select.maxCount)
            print("option type sample:", [getattr(o, "type", None) for o in obs.select.option[:10]])
        return obs
    finally:
        try:
            battle_finish()
        except Exception:
            pass

ENGINE_EDA_OBS = run_optional_engine_eda()





,card_type,size
0,0,1056
1,1,77
2,2,27
3,3,61
4,4,26
5,5,8
6,6,12


,hp
count,1267.000000
mean,102.257301
std,78.495651
min,0.000000
25%,60.000000
50%,90.000000
75%,130.000000
max,380.000000


battle_start: OK
obs.current is None: False
obs.select is None: False
select context: 41
select.option count: 2
min/max: 1 1
option type sample: [1, 2]


In [15]:
# Curated from the four official sample decks (card ids + counts). Used for a
# structure comparison only - no engine required.
SAMPLE_ARCHETYPES = {
    "Mega Lucario ex (Fighting)": {
        "axis": "Fighting multi-attacker; Riolu->Mega Lucario ex core, Hariyama/Solrock backup",
        "pokemon": [(673, 2), (674, 2), (675, 2), (676, 3), (677, 3), (678, 4)],
        "trainer": [(1102, 4), (1123, 2), (1141, 4), (1142, 4), (1152, 4), (1159, 1),
                     (1182, 2), (1192, 4), (1227, 4), (1252, 2)],
        "energy": [(6, 13)],
        "prize_liability": [("Mega Lucario ex", 3)],
    },
    "Dragapult ex (Fire/Psychic spread)": {
        "axis": "Stage-2 spread/snipe; Crushing Hammer disruption; 2-color",
        "pokemon": [(119, 4), (120, 4), (121, 3), (140, 1), (184, 1), (235, 2), (1071, 1)],
        "trainer": [(1079, 2), (1080, 1), (1086, 4), (1097, 2), (1120, 4), (1121, 4),
                     (1152, 3), (1156, 1), (1182, 3), (1198, 4), (1210, 2), (1227, 4), (1256, 2)],
        "energy": [(2, 4), (5, 4)],
        "prize_liability": [("Dragapult ex", 2), ("Fezandipiti ex", 2),
                             ("Latias ex", 2), ("Meowth ex", 2)],
    },
    "Iono Bellibolt ex (Lightning)": {
        "axis": "Lightning engine; Bellibolt ex acceleration; Iono support package",
        "pokemon": [(265, 3), (268, 3), (269, 3), (270, 3), (271, 3)],
        "trainer": [(1086, 3), (1097, 2), (1110, 1), (1118, 1), (1121, 3), (1152, 2),
                     (1227, 4), (1233, 4), (1254, 3)],
        "energy": [(4, 22)],
        "prize_liability": [("Bellibolt ex", 2)],
    },
    "Mega Abomasnow ex (Water)": {
        "axis": "Water beatdown; minimal tech; 34 basic energy",
        "pokemon": [(721, 2), (722, 4), (723, 4)],
        "trainer": [(1121, 4), (1126, 1), (1192, 4), (1227, 4), (1262, 3)],
        "energy": [(3, 34)],
        "prize_liability": [("Mega Abomasnow ex", 3), ("Kyogre", 1)],
    },
}


def _sum_counts(pairs):
    return sum(c for _, c in pairs)


arch_rows = []
for name, d in SAMPLE_ARCHETYPES.items():
    pk, tr, en = _sum_counts(d["pokemon"]), _sum_counts(d["trainer"]), _sum_counts(d["energy"])
    max_prize = max((p for _, p in d["prize_liability"]), default=1)
    n_two_prize_plus = sum(1 for _, p in d["prize_liability"] if p >= 2)
    arch_rows.append({
        "archetype": name,
        "pokemon": pk, "trainer": tr, "energy": en, "total": pk + tr + en,
        "unique_pokemon_lines": len(d["pokemon"]),
        "max_prize_given_up": max_prize,
        "ex_megaex_lines": n_two_prize_plus,
        "axis": d["axis"],
    })

if pd is not None:
    arch_df = pd.DataFrame(arch_rows)
    display(arch_df[["archetype", "pokemon", "trainer", "energy", "total",
                     "unique_pokemon_lines", "max_prize_given_up", "ex_megaex_lines"]])
else:
    for r in arch_rows:
        print(r)

print()
print("Takeaways:")
print("- Lucario is the field default -> mirror-heavy; edge there comes mostly from not crashing.")
print("- Abomasnow is the simplest shell (1 attacker line, 34 energy) -> good fixed opponent for eval.")
print("- Dragapult carries the most prize liability (4 two-prize+ lines) but has spread/disruption upside.")
print("- Bellibolt is the leanest engine deck. For Phase-2 differentiation, the non-Lucario space is wide open.")


,archetype,pokemon,trainer,energy,total,unique_pokemon_lines,max_prize_given_up,ex_megaex_lines
0,Mega Lucario ex (Fighting),16,31,13,60,6,3,1
1,Dragapult ex (Fire/Psychic spread),16,36,8,60,7,2,4
2,Iono Bellibolt ex (Lightning),15,23,22,60,5,2,1
3,Mega Abomasnow ex (Water),10,16,34,60,3,3,1



Takeaways:
- Lucario is the field default -> mirror-heavy; edge there comes mostly from not crashing.
- Abomasnow is the simplest shell (1 attacker line, 34 energy) -> good fixed opponent for eval.
- Dragapult carries the most prize liability (4 two-prize+ lines) but has spread/disruption upside.
- Bellibolt is the leanest engine deck. For Phase-2 differentiation, the non-Lucario space is wide open.


In [16]:
def run_action_space_eda(n_games=5, max_states=4000):
    """Roll out a few random-vs-random games and tally the action space + hidden info."""
    if CG_DIR is None:
        print("skip action-space EDA: cg/ unavailable in this runtime.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip action-space EDA: cg import failed:", repr(e))
        return None

    from collections import Counter
    deck = load_deck_ids()
    rnd = random_legal_agent_factory(deck)
    opt_types, contexts = Counter(), Counter()
    option_lens, max_counts = [], []
    hidden_checked = False
    states = 0

    for _ in range(n_games):
        try:
            obs, start = battle_start(deck, deck)
        except Exception as e:
            print("battle_start failed:", repr(e))
            break
        if getattr(start, "errorPlayer", -1) is not None and getattr(start, "errorPlayer", -1) >= 0:
            battle_finish()
            continue
        steps = 0
        while obs["current"]["result"] < 0 and states < max_states:
            sel = obs.get("select")
            if sel is not None:
                opts = sel.get("option") or []
                option_lens.append(len(opts))
                max_counts.append(sel.get("maxCount"))
                contexts[str(sel.get("context"))] += 1
                for o in opts:
                    opt_types[str(o.get("type"))] += 1
                states += 1
            if not hidden_checked and obs.get("current") is not None:
                try:
                    cur = obs["current"]
                    players = cur.get("players", [])
                    if len(players) == 2:
                        me = cur.get("yourIndex", 0)
                        opp = players[1 - me]
                        print("Hidden-information check (grounds Phase-2 belief modeling):")
                        print("  opponent handCount   :", opp.get("handCount"))
                        print("  opponent 'hand' field:", "present" if opp.get("hand") else "absent/empty (count-only)")
                        print("  my prize entries     :", players[me].get("prize", [])[:3], "(face-down should be None)")
                        hidden_checked = True
                except Exception as e:
                    print("  hidden-info check skipped:", repr(e))
                    hidden_checked = True
            obs = battle_select(rnd(obs))
            steps += 1
            if steps > MAX_STEPS_PER_GAME:
                break
        battle_finish()

    print()
    print("sampled decision states:", states)
    print("OptionType frequency   :", dict(opt_types.most_common()))
    print("SelectContext frequency:", dict(contexts.most_common()))
    if pd is not None and option_lens:
        display(pd.Series(option_lens).describe().to_frame("len(option) per decision"))
        clean_max = [m for m in max_counts if m is not None]
        if clean_max:
            display(pd.Series(clean_max).describe().to_frame("maxCount per decision"))
    return {"opt_types": dict(opt_types), "contexts": dict(contexts), "states": states}


ACTION_SPACE_EDA = run_action_space_eda(n_games=5)


Hidden-information check (grounds Phase-2 belief modeling):
  opponent handCount   : 0
  opponent 'hand' field: absent/empty (count-only)
  my prize entries     : [] (face-down should be None)

sampled decision states: 671
OptionType frequency   : {'7': 1249, '8': 871, '3': 751, '14': 464, '13': 150, '10': 140, '9': 105, '12': 77, '6': 30, '1': 10, '2': 10, '0': 6}
SelectContext frequency: {'0': 464, '7': 74, '3': 42, '8': 21, '30': 16, '21': 12, '1': 10, '4': 8, '2': 7, '41': 5, '43': 5, '22': 5, '38': 2}


,len(option) per decision
count,671.000000
mean,5.757079
std,4.268840
min,1.000000
25%,3.000000
50%,5.000000
75%,8.000000
max,23.000000


,maxCount per decision
count,671.000000
mean,1.017884
std,0.171844
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000


In [17]:
def validate_submission_bundle(path: Path, require_cg: bool = True) -> list[str]:
    path = Path(path)
    assert path.exists(), f"missing archive: {path}"
    assert path.name.endswith(".tar.gz"), path

    with tarfile.open(path, "r:gz") as tar:
        names = tar.getnames()
        name_set = set(names)

        errors = []
        if "main.py" not in name_set:
            errors.append("main.py must be at archive root")
        if "deck.csv" not in name_set:
            errors.append("deck.csv must be at archive root")
        if any(name.endswith("submission.csv") or name == "submission.csv" for name in names):
            errors.append("submission.csv should not be included for this agent competition")
        forbidden = [n for n in names if n.endswith((".ipynb", ".pyc", ".pyo")) or "__pycache__" in n or n.startswith("benchmark_logs/")]
        if forbidden:
            errors.append(f"forbidden artifacts included: {forbidden[:5]}")

        if "deck.csv" in name_set:
            raw = tar.extractfile("deck.csv").read().decode().splitlines()
            deck = [int(line.strip()) for line in raw if line.strip()]
            if len(deck) != 60:
                errors.append(f"deck.csv must contain 60 integer IDs, got {len(deck)}")

        if "main.py" in name_set:
            main = tar.extractfile("main.py").read().decode()
            compile(main, "main.py", "exec")
            if "def agent" not in main:
                errors.append("main.py must expose agent(obs_dict)")
            if "from cg.api" in main and require_cg:
                required_cg = {"cg/api.py", "cg/game.py"}
                missing_cg = sorted(required_cg - name_set)
                has_binary = any(n in name_set for n in ["cg/libcg.so", "cg/cg.dll"])
                if missing_cg:
                    errors.append(f"missing cg files: {missing_cg}")
                if not has_binary:
                    errors.append("missing cg binary: expected cg/libcg.so or cg/cg.dll")

        if errors:
            raise AssertionError("\n".join(errors))

    print("submission contract OK")
    print("archive files:", len(names))
    for name in sorted(names)[:20]:
        print(" -", name)
    if len(names) > 20:
        print(" ...")
    return names




In [18]:
# ============================================================================
# Phase-1 VALIDATION GATE  (battle_start based, with fallback diagnostics)
# ----------------------------------------------------------------------------
# Why battle_start and not the env wrapper: it runs in-process (so we can read the
# agent's _DIAG), gives an unambiguous winner index, and matches the official
# samples. The gate checks engine errors, win-rate, policy fallback, and search
# failure rate when search is enabled. A policy can be stable while search is
# silently failing, so these signals are tracked separately.
# ============================================================================
def play_game(battle_start, battle_select, battle_finish, deck0, deck1, agent0, agent1):
    """Return (result, steps, error). result: 2 draw / 0|1 winner seat / None on error."""
    try:
        obs, start = battle_start(deck0, deck1)
    except Exception as e:
        return None, 0, "battle_start raised: " + repr(e)
    err = getattr(start, "errorPlayer", -1)
    if err is not None and err >= 0:
        try:
            battle_finish()
        except Exception:
            pass
        return None, 0, "deck error: player=%s type=%s" % (err, getattr(start, "errorType", "?"))
    steps = 0
    try:
        while obs["current"]["result"] < 0:
            seat = obs["current"]["yourIndex"]
            sel = (agent0 if seat == 0 else agent1)(obs)
            obs = battle_select(sel)
            steps += 1
            if steps > MAX_STEPS_PER_GAME:
                battle_finish()
                return None, steps, "exceeded MAX_STEPS_PER_GAME"
        result = obs["current"]["result"]
    except Exception as e:
        try:
            battle_finish()
        except Exception:
            pass
        return None, steps, "game loop raised: " + repr(e)
    battle_finish()
    return result, steps, None


def run_validation_gate(n_games, enforce=True):
    if CG_DIR is None:
        print("skip validation gate: cg/ unavailable in this runtime.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip validation gate: cg import failed:", repr(e))
        return None

    deck = load_deck_ids()
    mod = import_generated_module()
    agent = mod.agent
    rnd = random_legal_agent_factory(deck)
    report = {}

    # ---- Mirror: agent vs a copy of itself (what the ladder validation does). ----
    mod.diag_reset()
    mirror_errors = 0
    for g in range(n_games):
        res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, agent, agent)
        if err is not None or res is None:
            mirror_errors += 1
            if mirror_errors <= 5:
                print("  mirror engine error:", err)
    md = mod.diag_snapshot()
    report["mirror_errors"] = mirror_errors
    report["mirror_fallback_rate"] = md["fallback_rate"]
    report["mirror_search_fail_rate"] = md.get("search_fail_rate", 0.0)
    report["mirror_decisions"] = md["decisions"]
    report["mirror_top_errors"] = sorted(md.get("policy_errors", md["errors"]).items(), key=lambda kv: -kv[1])[:6]
    report["mirror_top_search_errors"] = sorted(md.get("search_errors", {}).items(), key=lambda kv: -kv[1])[:6]

    # ---- vs legal-random, alternating seat. ----
    mod.diag_reset()
    w = d = l = 0
    for g in range(n_games):
        if g % 2 == 0:
            res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, agent, rnd)
            my_seat = 0
        else:
            res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, rnd, agent)
            my_seat = 1
        if err is not None or res is None:
            continue
        if res == 2:
            d += 1
        elif res == my_seat:
            w += 1
        else:
            l += 1
    played = w + d + l
    winrate = w / played if played else 0.0
    rd = mod.diag_snapshot()
    report["vs_random_w_d_l"] = (w, d, l)
    report["vs_random_winrate"] = winrate
    report["vs_random_fallback_rate"] = rd["fallback_rate"]
    report["vs_random_search_fail_rate"] = rd.get("search_fail_rate", 0.0)
    report["vs_random_top_search_errors"] = sorted(rd.get("search_errors", {}).items(), key=lambda kv: -kv[1])[:6]

    # ---- Report ----
    print("=" * 64)
    print("PHASE-1 VALIDATION GATE   (n_games per matchup = %d)" % n_games)
    print("-" * 64)
    print("Mirror     : errors=%d/%d   fallback_rate=%.3f   search_fail_rate=%.3f   decisions=%d"
          % (mirror_errors, n_games, report["mirror_fallback_rate"],
             report["mirror_search_fail_rate"], report["mirror_decisions"]))
    if report["mirror_top_errors"]:
        print("  >>> the policy is FALLING BACK on these exceptions -- fix on the live SDK:")
        for msg, cnt in report["mirror_top_errors"]:
            print("      [%5d x] %s" % (cnt, msg))
    print("vs random  : W/D/L=%d/%d/%d   winrate=%.3f   fallback_rate=%.3f   search_fail_rate=%.3f"
          % (w, d, l, winrate, report["vs_random_fallback_rate"], report["vs_random_search_fail_rate"]))
    print("=" * 64)

    fallback_rate = max(report["mirror_fallback_rate"], report["vs_random_fallback_rate"])
    search_fail_rate = max(report["mirror_search_fail_rate"], report["vs_random_search_fail_rate"])
    search_ok = (not getattr(mod, "USE_SEARCH", False)) or (search_fail_rate <= SEARCH_FAIL_GATE)
    gate_pass = (
        (mirror_errors == 0)
        and (winrate >= WINRATE_GATE)
        and (fallback_rate <= FALLBACK_GATE)
        and search_ok
    )
    if gate_pass:
        print("GATE PASS  PASS  safe-to-submit floor met.")
    else:
        print("GATE FAIL  FAIL  do NOT submit yet:")
        if mirror_errors != 0:
            print("  - mirror produced engine errors (expected 0).")
        if winrate < WINRATE_GATE:
            print("  - winrate %.3f < gate %.2f." % (winrate, WINRATE_GATE))
        if fallback_rate > FALLBACK_GATE:
            print("  - fallback_rate %.3f > gate %.2f: the policy is silently broken on the live SDK"
                  % (fallback_rate, FALLBACK_GATE))
            print("    (0 errors here means the try/except swallowed the bug -- see the exception list above).")
        if not search_ok:
            print("  - search_fail_rate %.3f > gate %.2f while search is enabled."
                  % (search_fail_rate, SEARCH_FAIL_GATE))
            for msg, cnt in (report["mirror_top_search_errors"] + report["vs_random_top_search_errors"])[:6]:
                print("      search [%5d x] %s" % (cnt, msg))
    report["gate_pass"] = gate_pass
    if enforce:
        assert gate_pass, "Phase-1 validation gate failed (see breakdown above)."
    return report


# Raise VALIDATION_GAMES toward a few hundred for a confident pre-submission check.
VALIDATION_GAMES = N_SMOKE_GAMES   # 40 from the config cell
GATE_REPORT = run_validation_gate(n_games=VALIDATION_GAMES, enforce=ENFORCE_GATE)

PHASE-1 VALIDATION GATE   (n_games per matchup = 40)
----------------------------------------------------------------
Mirror     : errors=0/40   fallback_rate=0.000   search_fail_rate=0.000   decisions=5114
vs random  : W/D/L=40/0/0   winrate=1.000   fallback_rate=0.000   search_fail_rate=0.000
GATE PASS  PASS  safe-to-submit floor met.


In [19]:
# ============================================================================
# Search A/B  +  attack/branch firing audit
#   (1) Does search actually beat the greedy policy head-to-head? That is the
#       only thing that moves you off the ~600 mirror plateau.
#   (2) Silent dead-ends: a wrong MEGA_BRAVE attackId (983) throws NO exception,
#       it just makes the 270-dmg attack never fire. We print the REAL attackIds
#       seen on a Mega Lucario active so you can fix the constant.
# ============================================================================
def run_search_ab_and_audit(ab_games=20, audit_games=8):
    if CG_DIR is None:
        print("skip: cg/ unavailable; run on Kaggle with cg-lib attached.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip: cg import failed:", repr(e))
        return None

    deck = load_deck_ids()
    mod = import_generated_module()

    # ---- (1) Search vs greedy, same deck, alternating seat ----
    if not getattr(mod, "_SEARCH_AVAILABLE", False):
        print("NOTE: cg search API not importable in this build -> search disabled, A/B skipped.")
    else:
        def toggle_agent(use_search):
            def a(obs_dict):
                mod.USE_SEARCH = use_search
                return mod.agent(obs_dict)
            return a
        search_a, greedy_a = toggle_agent(True), toggle_agent(False)
        mod.diag_reset()
        w = d = l = 0
        for g in range(ab_games):
            if g % 2 == 0:
                res, _, err = play_game(battle_start, battle_select, battle_finish, deck, deck, search_a, greedy_a)
                seat = 0
            else:
                res, _, err = play_game(battle_start, battle_select, battle_finish, deck, deck, greedy_a, search_a)
                seat = 1
            if err is not None or res is None:
                continue
            if res == 2:
                d += 1
            elif res == seat:
                w += 1
            else:
                l += 1
        played = w + d + l
        wr = w / played if played else 0.0
        snap = mod.diag_snapshot()
        print("=" * 64)
        print("SEARCH A/B   (search agent vs greedy agent, identical deck)")
        print("  search W/D/L = %d/%d/%d   winrate = %.3f   (n=%d)" % (w, d, l, wr, played))
        print("  search_used=%d  search_failed=%d  search_fail_rate=%.3f"
              % (snap.get("search_used", 0), snap.get("search_failed", 0), snap.get("search_fail_rate", 0.0)))
        fail_rate = snap.get("search_fail_rate", 0.0)
        if fail_rate > SEARCH_FAIL_GATE:
            print("  -> search is FAILING on the live engine; keep greedy until this is fixed.")
            for msg, cnt in sorted(snap.get("search_errors", snap.get("errors", {})).items(), key=lambda kv: -kv[1])[:5]:
                print("      [%4d x] %s" % (cnt, msg))
        elif wr > 0.55 and played >= 200:
            print("  -> search HELPS under the stricter gate; it is eligible for a search-enabled variant.")
        else:
            print("  -> search is not yet proven; keep the default greedy profile.")
        print("=" * 64)
        mod.USE_SEARCH = False

    # ---- (2) Attack / branch firing audit (greedy mirror) ----
    mod.USE_SEARCH = False
    mod.diag_reset()
    for _ in range(audit_games):
        play_game(battle_start, battle_select, battle_finish, deck, deck, mod.agent, mod.agent)
    snap = mod.diag_snapshot()
    print("ATTACK / BRANCH FIRING AUDIT   (greedy mirror, %d games)" % audit_games)
    print("-" * 64)
    print("MEGA_BRAVE constant in agent =", getattr(mod, "MEGA_BRAVE", "?"),
          "(should match a real Mega Lucario attackId below)")
    for active_id, aids in sorted(snap.get("attack_opts_by_active", {}).items()):
        nm = "Mega Lucario ex(678)" if active_id == 678 else str(active_id)
        print("  active %-22s legal attackIds seen: %s" % (nm, aids))
    print("attackIds actually CHOSEN :", snap.get("attack_ids_chosen", {}))
    print("MEGA_BRAVE seen as option :", snap.get("mega_brave_present", 0), "times")
    print("chosen option-type counts :", snap.get("chosen_types", {}))
    if snap.get("mega_brave_present", 0) == 0:
        print("  !! MEGA_BRAVE was NEVER a legal attack -> 983 is almost certainly wrong.")
        print("     Replace MEGA_BRAVE with the real Mega Lucario(678) attackId listed above.")
    print("=" * 64)
    return snap


AUDIT = run_search_ab_and_audit(ab_games=20, audit_games=8)

SEARCH A/B   (search agent vs greedy agent, identical deck)
  search W/D/L = 11/0/9   winrate = 0.550   (n=20)
  search_used=0  search_failed=8  search_fail_rate=1.000
  -> search is FAILING on the live engine; keep greedy until this is fixed.
      [   8 x] TypeError: search_begin() missing 6 required positional arguments: 'your_deck', 'your_prize', 'opponent_deck', 'opponent_prize', 'opponent_hand', and 'opponent_active'
ATTACK / BRANCH FIRING AUDIT   (greedy mirror, 8 games)
----------------------------------------------------------------
MEGA_BRAVE constant in agent = 983 (should match a real Mega Lucario attackId below)
  active 673                    legal attackIds seen: [976, 977]
  active 674                    legal attackIds seen: [978]
  active 676                    legal attackIds seen: [980]
  active 677                    legal attackIds seen: [981]
  active Mega Lucario ex(678)   legal attackIds seen: [982, 983]
attackIds actually CHOSEN : {981: 4, 982: 33, 980: 11, 98

In [20]:
if WRITE_SUBMISSION:
    required = [BUILD_DIR / "main.py", BUILD_DIR / "deck.csv"]
    for p in required:
        assert p.exists(), p

    cg_dir = BUILD_DIR / "cg"
    if REQUIRE_CG_IN_ARCHIVE and not cg_dir.exists():
        raise FileNotFoundError(
            "cg/ was not copied. Attach the competition cg-lib input, rerun the cg copy cell, then create submission.tar.gz."
        )

    with tarfile.open(SUBMISSION_PATH, "w:gz") as tar:
        tar.add(BUILD_DIR / "main.py", arcname="main.py")
        tar.add(BUILD_DIR / "deck.csv", arcname="deck.csv")
        if cg_dir.exists():
            for p in sorted(cg_dir.rglob("*")):
                if p.is_file() and "__pycache__" not in p.parts and not p.name.endswith((".pyc", ".pyo")):
                    tar.add(p, arcname=str(p.relative_to(BUILD_DIR)))

    print("created", SUBMISSION_PATH)
    archive_names = validate_submission_bundle(SUBMISSION_PATH, require_cg=REQUIRE_CG_IN_ARCHIVE)
else:
    print("WRITE_SUBMISSION=False; archive not created.")




created /kaggle/working/submission.tar.gz
submission contract OK
archive files: 9
 - cg/__init__.py
 - cg/api.py
 - cg/cg.dll
 - cg/game.py
 - cg/libcg.so
 - cg/sim.py
 - cg/utils.py
 - deck.csv
 - main.py


## Strategic Follow-Up

Next controlled experiments:

1. matchup matrix across official sample agents,
2. ablations for draw suppression, Boss/Switch planning, target scoring, and search budget,
3. deck-variant tests against Dragapult, Iono/Bellibolt, and Mega Abomasnow,
4. shallow search only in high-leverage tactical contexts,
5. belief-guided search after rule-policy diagnostics are stable.